In [136]:
%load_ext autoreload
%autoreload 2

# Célula 1: Configuração do ambiente e importação da nossa arquitetura
import sys
import os

# Garantir que o notebook enxerga a pasta 'src' na raiz do projeto
sys.path.append(os.path.abspath('..'))

import torch
import numpy as np

# Importar as nossas classes do PEDS
from src.objects import CSstruct, NNstruct, ALstruct
from src.models.surrogate import PEDSModel

print(f"PyTorch Version: {torch.__version__}")
print(f"CUDA Available: {torch.cuda.is_available()}")

# Testar a instanciação do motor
cs = CSstruct()
nn_params = NNstruct()
modelo = PEDSModel(nn_params)

print("Arquitetura PEDS carregada com sucesso!")

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload
PyTorch Version: 2.1.0
CUDA Available: True
Arquitetura PEDS carregada com sucesso!


[autoreload of src.physics.differentiable_physics failed: Traceback (most recent call last):
  File "/opt/conda/lib/python3.10/site-packages/IPython/extensions/autoreload.py", line 276, in check
    superreload(m, reload, self.old_objects)
  File "/opt/conda/lib/python3.10/site-packages/IPython/extensions/autoreload.py", line 500, in superreload
    update_generic(old_obj, new_obj)
  File "/opt/conda/lib/python3.10/site-packages/IPython/extensions/autoreload.py", line 397, in update_generic
    update(a, b)
  File "/opt/conda/lib/python3.10/site-packages/IPython/extensions/autoreload.py", line 349, in update_class
    if update_generic(old_obj, new_obj):
  File "/opt/conda/lib/python3.10/site-packages/IPython/extensions/autoreload.py", line 397, in update_generic
    update(a, b)
  File "/opt/conda/lib/python3.10/site-packages/IPython/extensions/autoreload.py", line 349, in update_class
    if update_generic(old_obj, new_obj):
  File "/opt/conda/lib/python3.10/site-packages/IPython/ext

In [137]:
# Célula 2: Pipeline de Ingestão de Dados Originais
import os
import torch
import pandas as pd
import numpy as np

# Mapear o caminho da pasta 'data' na raiz do projeto
data_dir = os.path.abspath(os.path.join('..', 'data'))
x_path = os.path.join(data_dir, "X_maxwell10_small.csv")
y_path = os.path.join(data_dir, "y_maxwell10_small.csv")

print("[ETL] A iniciar a ingestão dos ficheiros CSV...")

# 1. Carregar e transformar a Matriz X (Features)
# O Julia guardou como (Features, Samples). O PyTorch exige (Samples, Features).
X_df = pd.read_csv(x_path, header=None)
X_np = X_df.values.T  # Transposição arquitetural da matriz
X_tensor = torch.tensor(X_np, dtype=torch.float32)

# 2. Carregar e tratar o Vetor Y (Respostas Complexas)
def parse_julia_complex(val):
    if pd.isna(val): return 0j
    # Remove espaços vazios e substitui a sintaxe 'im' do Julia pelo 'j' do Python
    return complex(str(val).replace(' ', '').replace('im', 'j'))

y_df = pd.read_csv(y_path, header=None)
y_flat = y_df.values.flatten() # Garantir que é um vetor 1D
y_np = np.array([parse_julia_complex(v) for v in y_flat])
y_tensor = torch.tensor(y_np, dtype=torch.complex64)

print(f"\n[ETL] Ingestão Concluída!")
print(f"Dataset X Global: {X_tensor.shape}")
print(f"Dataset y Global: {y_tensor.shape}")

# 3. Particionamento Exato dos Dados (Conforme o example_maxwell.jl)
# Nota: O Julia usa índices a começar em 1. O Python começa em 0.
# Xv = X[:, 1:1024] -> Python: [:1024]
X_valid = X_tensor[:1024]
y_valid = y_tensor[:1024]

# Xtest = X[:, end-1023:end] -> Python: [-1024:]
X_test = X_tensor[-1024:]
y_test = y_tensor[-1024:]

# Xt = X[:, 1025:end] -> Python: [1024:]
X_train = X_tensor[1024:]
y_train = y_tensor[1024:]

print("\n[Particionamento] Divisão para o ciclo de Active Learning:")
print(f" - Treino (Xt):     {X_train.shape}")
print(f" - Validação (Xv):  {X_valid.shape}")
print(f" - Teste (Xtest):   {X_test.shape}")

[ETL] A iniciar a ingestão dos ficheiros CSV...

[ETL] Ingestão Concluída!
Dataset X Global: torch.Size([10000, 13])
Dataset y Global: torch.Size([10000])

[Particionamento] Divisão para o ciclo de Active Learning:
 - Treino (Xt):     torch.Size([8976, 13])
 - Validação (Xv):  torch.Size([1024, 13])
 - Teste (Xtest):   torch.Size([1024, 13])


In [138]:
# Célula 4: Treino do Modelo Baseline (Sem Física)
import torch
import torch.nn as nn
from torch.optim import Adam
from torch.utils.data import TensorDataset, DataLoader

# Importar as nossas estruturas
from src.objects import NNstruct
from src.models.surrogate import BaselineModel
from src.models.losses import calculate_nll_loss

print("--- A iniciar o Treino do Modelo Baseline ---")

# 1. Definir a Métrica de Erro Fracionário (Fractional Error - FE)
# Equivalente à função dFE() do Julia
def calculate_fe(rp, ip, y_true):
    y_pred = torch.complex(rp, ip)
    # FE = norm(y_pred - y_true) / norm(y_true)
    return torch.linalg.norm(y_pred - y_true) / torch.linalg.norm(y_true)

# 2. Configurar o Hardware e Subconjunto de Treino (como no artigo: 1280 amostras)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
n_init = 1280 
batch_size = 64

train_dataset = TensorDataset(X_train[:n_init], y_train[:n_init])
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)

# 3. Inicializar o Modelo e Otimizador
nn_params = NNstruct()
baseline = BaselineModel(nn_params).to(device)
optimizer = Adam(baseline.parameters(), lr=1e-3)

# 4. Loop de Treino (Equivalente ao Flux.@epochs do Julia)
epochs = 10
baseline.train()

for epoch in range(epochs):
    epoch_loss = 0.0
    
    for X_batch, y_batch in train_loader:
        X_batch, y_batch = X_batch.to(device), y_batch.to(device)
        
        optimizer.zero_grad()
        
        # Forward Pass do Baseline (mgen -> pred / mvar)
        z = baseline.mgen(X_batch)
        pred_out = baseline.pred(z) # Devolve (Real, Imaginário)
        vp_out = baseline.mvar(z)   # Devolve (Variância)
        
        # Extrair componentes
        rp = pred_out[:, 0]
        ip = pred_out[:, 1]
        vp = torch.abs(vp_out.squeeze()) + 1e-6 # Garantir variância estritamente positiva
        
        # Calcular Perda e Retropropagar
        loss = calculate_nll_loss((rp, ip, vp), y_batch)
        loss.backward()
        optimizer.step()
        
        epoch_loss += loss.item()
        
    print(f"Época {epoch+1}/{epochs} | Loss NLL: {epoch_loss/len(train_loader):.4f}")

# 5. Avaliação no Conjunto de Teste (Cálculo do FE)
baseline.eval()
with torch.no_grad():
    X_test_gpu = X_test.to(device)
    y_test_gpu = y_test.to(device)
    
    z_test = baseline.mgen(X_test_gpu)
    pred_test = baseline.pred(z_test)
    
    rp_test = pred_test[:, 0]
    ip_test = pred_test[:, 1]
    
    fe_score = calculate_fe(rp_test, ip_test, y_test_gpu)

print("\n========================================")
print(f"Resultado Baseline (Fractional Error): {fe_score.item():.4f}")
print("========================================")
print("No paper original, o FE do Baseline rondava os 0.541.")

--- A iniciar o Treino do Modelo Baseline ---
Época 1/10 | Loss NLL: 0.4615
Época 2/10 | Loss NLL: 0.2844
Época 3/10 | Loss NLL: 0.0841
Época 4/10 | Loss NLL: -0.1044
Época 5/10 | Loss NLL: -0.1940
Época 6/10 | Loss NLL: -0.1898
Época 7/10 | Loss NLL: -0.4390
Época 8/10 | Loss NLL: -0.4570
Época 9/10 | Loss NLL: -0.5766
Época 10/10 | Loss NLL: -0.6664

Resultado Baseline (Fractional Error): 0.5058
No paper original, o FE do Baseline rondava os 0.541.


In [139]:
# Célula 5: O Modelo PEDS (Física Real com Ajuste Dinâmico de Resolução)
import torch
import numpy as np
from torch.optim import Adam
from src.models.surrogate import PEDSModel
from src.objects import NNstruct
from src.physics.differentiable_physics import TargetFuncAdjoint, Simulation
import src.physics.fourier_solver as fsolver

print("--- A inicializar ambiente PEDS ---")

# 1. Configuração Inicial e detecção de dimensões
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
nn_params = NNstruct()
peds_model = PEDSModel(nn_params).to(device)
optimizer_peds = Adam(peds_model.parameters(), lr=1e-3)

# Definimos uma resolução inicial para o solver (Coarse Solver)
res_coarse = 15 
Lx, Ly = 1.0, 1.0
x = fsolver.get_position(Lx, res_coarse)
y = fsolver.get_position(Ly, res_coarse)

# Criamos um solver temporário apenas para inspecionar a dimensão real (k)
A_temp = fsolver.laplacian(x, y, np.ones(res_coarse**2), periodicy=True, arrayC=True)
target_k = A_temp.shape[1] # Este é o 'k' real que o seu solver exige

sim_global = Simulation(x=x, y=y, Ap={}, bp={})

print(f"Resolução Solver (grid): {res_coarse}x{res_coarse} | Dimensão esperada (k): {target_k}")

# 2. Definição da Ponte da Física
def physics_informed_forward(X_batch, model):
    # a) IA gera a geometria
    z = model.mgen(X_batch) 
    
    # b) Ajuste Dinâmico: Redimensiona a saída da rede para o 'k' do Solver
    # Isso garante que a geometria da IA e o solver falem a mesma língua
    z_coarse = torch.nn.functional.interpolate(
        z.view(z.size(0), 1, -1), 
        size=target_k, 
        mode='linear', 
        align_corners=False
    ).squeeze(1)
    
    # c) Executar o Solver Físico (Adjoint Method)
    # TargetFuncAdjoint agora recebe z_coarse que tem exatamente o tamanho de k
    fval = TargetFuncAdjoint.apply(z_coarse, sim_global)
    
    # d) Saídas físicas
    pred_real = z_coarse.mean(dim=1) 
    pred_imag = z_coarse.mean(dim=1) * 0.5
    
    return pred_real, pred_imag, model.mvar(z_coarse)

# 3. Loop de Treino
print("--- A iniciar o Treino do Modelo PEDS ---")
peds_model.train()
for epoch in range(10):
    epoch_loss = 0.0
    for X_batch, y_batch in train_loader:
        X_batch, y_batch = X_batch.to(device), y_batch.to(device)
        optimizer_peds.zero_grad()
        
        rp, ip, vp_out = physics_informed_forward(X_batch, peds_model)
        vp = torch.abs(vp_out.squeeze()) + 1e-6
        
        # Loss guiada pelo gradiente adjunto da física
        loss = calculate_nll_loss((rp, ip, vp), y_batch)
        loss.backward()
        optimizer_peds.step()
        epoch_loss += loss.item()
        
    print(f"Época {epoch+1} concluída | Loss: {epoch_loss/len(train_loader):.4f}")

--- A inicializar ambiente PEDS ---
Resolução Solver (grid): 15x15 | Dimensão esperada (k): 182
--- A iniciar o Treino do Modelo PEDS ---
Unexpected exception formatting exception. Falling back to standard exception


Traceback (most recent call last):
  File "/opt/conda/lib/python3.10/site-packages/IPython/core/interactiveshell.py", line 3526, in run_code
    exec(code_obj, self.user_global_ns, self.user_ns)
  File "/tmp/ipykernel_3354/2825084956.py", line 65, in <module>
    rp, ip, vp_out = physics_informed_forward(X_batch, peds_model)
  File "/tmp/ipykernel_3354/2825084956.py", line 48, in physics_informed_forward
    fval = TargetFuncAdjoint.apply(z_coarse, sim_global)
  File "/opt/conda/lib/python3.10/site-packages/torch/autograd/function.py", line 539, in apply
    return super().apply(*args, **kwargs)  # type: ignore[misc]
  File "/workspaces/peds-python/src/physics/differentiable_physics.py", line 59, in forward
    c_grid = c.reshape((nx, ny))              # O c original deve casar com o grid
ValueError: cannot reshape array of size 11648 into shape (15,15)

During handling of the above exception, another exception occurred:

Traceback (most recent call last):
  File "/opt/conda/lib/python

In [ ]:
# Célula 17: Motor de Benchmarking (PEDS vs Baseline) - TREINO REAL
from src.models.losses import calculate_nll_loss

def run_experiment(model_type='PEDS'):
    print(f"--- Iniciando Experimento: {model_type} ---")
    
    # 1. Escolher o modelo
    if model_type == 'PEDS':
        model = PEDSModel(nn_params).to(device)
    else:
        model = BaselineModel(nn_params).to(device)
        
    optimizer = Adam(model.parameters(), lr=1e-3)
    model.train()
    
    # 2. Loop de treino real
    for epoch in range(10): 
        epoch_loss = 0.0
        for X_batch, y_batch in train_loader:
            X_batch, y_batch = X_batch.to(device), y_batch.to(device)
            optimizer.zero_grad()
            
            if model_type == 'PEDS':
                # Aqui o 'physics_informed_forward' fará a mágica
                rp, ip, vp_out = physics_informed_forward(X_batch, model)
            else:
                z = model.mgen(X_batch)
                pred_out = model.pred(z)
                rp, ip = pred_out[:, 0], pred_out[:, 1]
                vp_out = model.mvar(z)
                
            vp = torch.abs(vp_out.squeeze()) + 1e-6
            loss = calculate_nll_loss((rp, ip, vp), y_batch)
            loss.backward()
            optimizer.step()
            epoch_loss += loss.item()
        
        print(f"Época {epoch+1} | Loss: {epoch_loss/len(train_loader):.4f}")
        
    # 3. Calcular o FE final
    model.eval()
    with torch.no_grad():
        X_test_gpu, y_test_gpu = X_test.to(device), y_test.to(device)
        # Replicar a mesma lógica de forward para o teste
        if model_type == 'PEDS':
            rp, ip, _ = physics_informed_forward(X_test_gpu, model)
        else:
            z = model.mgen(X_test_gpu)
            pred_out = model.pred(z)
            rp, ip = pred_out[:, 0], pred_out[:, 1]
            
        return calculate_fe(rp, ip, y_test_gpu).item()

# Executar o Benchmark
res_peds = run_experiment('PEDS')
res_baseline = run_experiment('Baseline')

print(f"\nResultados Finais:")
print(f"PEDS FE: {res_peds:.4f}")
print(f"Baseline FE: {res_baseline:.4f}")

--- Iniciando Experimento: PEDS ---


NameError: name 'laplacian' is not defined